# Task A — train on train + labelled validation, predict released test

This is the final-fit version of the current best Task A model. It combines the original 6,446 labelled training rows with the 806 released labelled validation rows, fits the Run 11 one-layer MuRIL + demojized TF-IDF/SVM blend, and writes predictions for the 806-row released test file.

The blend weight and threshold are refitted with nested OOF predictions on the combined labelled corpus before the final full-data fits. The output is for the newly released test file, not the closed CodaBench validation file. Expected runtime is roughly 4–6 hours on a Kaggle T4 ×2.

In [ ]:
import os, pathlib, shutil, subprocess, sys, zipfile, re, json
WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1", "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"', shell=True, check=True)
import numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from hastika.common.preprocessing import clean, dedupe_index
assert torch.cuda.is_available(), "enable a GPU"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen([str(x) for x in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh: fh.write(line)
    p.wait()
    if fh: fh.close()
    if p.returncode: raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Build the combined labelled corpus and stage the released test file

In [ ]:
from pathlib import Path
train_path = Path("data/raw/binary_train.csv")
val_path = Path("data/raw/hastika_binary_validation.csv")
test_path = Path("data/raw/hastika_binary_test.csv")
combined_path = Path("data/derived/task_a_train_plus_validation.csv")
combined_path.parent.mkdir(parents=True, exist_ok=True)
combined = pd.concat([pd.read_csv(train_path), pd.read_csv(val_path)], ignore_index=True)
combined = combined.drop_duplicates("id", keep="first").reset_index(drop=True)
assert combined["id"].is_unique and len(combined) == 7252
combined.to_csv(combined_path, index=False)
print("combined labelled rows:", len(combined), "test rows:", len(pd.read_csv(test_path)))

# The existing runners use canonical filenames. Back them up, stage the combined
# training data and released test comments, and restore them before packaging.
CANON_TRAIN = Path("data/raw/binary_train.csv")
CANON_TEST = Path("data/raw/binary_validation_inputs.csv")
shutil.copy2(CANON_TRAIN, CANON_TRAIN.with_suffix(".csv.before_final"))
shutil.copy2(CANON_TEST, CANON_TEST.with_suffix(".csv.before_final"))
shutil.copy2(combined_path, CANON_TRAIN)
shutil.copy2(test_path, CANON_TEST)


## 2. Refit the blend weight and threshold with combined-corpus OOF predictions

In [ ]:
SVM_OOF, MURIL_OOF = "a_trainval_svm_oof", "a_trainval_muril_oof"
run([sys.executable, "-u", "-m", "hastika.models.baseline_svm", "--task", "a", "--tag", SVM_OOF, "--demojize"], log=f"artifacts/logs/{SVM_OOF}.log")
run([sys.executable, "-u", "-m", "hastika.models.muril", "--tag", MURIL_OOF, "--model", "google/muril-base-cased", "--folds", "5", "--seeds", "42", "--epochs", "6", "--bs", "8", "--grad-accum", "2", "--eval-bs", "32", "--select", "last", "--reinit-layers", "1"], log=f"artifacts/logs/{MURIL_OOF}.log")
keep = dedupe_index(combined["Comment"].tolist(), combined["Label"].tolist(), "task A")
fit_df = combined.iloc[keep].reset_index(drop=True)
y = (fit_df["Label"] == "Hate").astype(int).to_numpy()
svm_oof = np.load(Path("artifacts/runs") / SVM_OOF / "oof_probs.npy")
muril_oof = np.load(Path("artifacts/runs") / MURIL_OOF / "oof_probs.npy")
assert svm_oof.shape == muril_oof.shape == (len(y), 2)
def score(w, t, idx):
    p = w * svm_oof[idx, 1] + (1 - w) * muril_oof[idx, 1]
    return f1_score(y[idx], p > t, average="macro")
grid_w, grid_t = np.arange(0, 1.001, .05), np.arange(.30, .71, .02)
picks, pred = [], np.zeros(len(y), dtype=int)
for tr, va in StratifiedKFold(5, shuffle=True, random_state=42).split(fit_df, y):
    inner_tr, inner_va = next(StratifiedKFold(4, shuffle=True, random_state=7).split(fit_df.iloc[tr], y[tr]))
    inner = tr[inner_va]
    w, t = max(((w, t) for w in grid_w for t in grid_t), key=lambda z: score(z[0], z[1], inner))
    picks.append((float(w), float(t)))
    p = w * svm_oof[va, 1] + (1 - w) * muril_oof[va, 1]
    pred[va] = p > t
W_SVM, THRESH = float(np.mean([x[0] for x in picks])), float(np.mean([x[1] for x in picks]))
print("nested OOF macro-F1:", f1_score(y, pred, average="macro"), "weight/threshold:", W_SVM, THRESH)


## 3. Fit the best Task A components on all combined labelled rows

In [ ]:
SVM_FULL, MURIL_FULL = "a_trainval_svm_full", "a_trainval_muril_full"
run([sys.executable, "-u", "-m", "hastika.models.baseline_svm", "--task", "a", "--tag", SVM_FULL, "--demojize", "--full-fit"], log=f"artifacts/logs/{SVM_FULL}.log")
run([sys.executable, "-u", "-m", "hastika.models.muril", "--tag", MURIL_FULL, "--model", "google/muril-base-cased", "--folds", "1", "--seeds", "42", "--epochs", "6", "--bs", "8", "--grad-accum", "2", "--eval-bs", "32", "--select", "last", "--reinit-layers", "1"], log=f"artifacts/logs/{MURIL_FULL}.log")
svm_p = np.load(Path("artifacts/runs") / SVM_FULL / "test_probs.npy")
muril_p = np.load(Path("artifacts/runs") / MURIL_FULL / "test_probs.npy")
test = pd.read_csv(test_path)
p = W_SVM * svm_p[:, 1] + (1 - W_SVM) * muril_p[:, 1]
pred_path = Path("artifacts/runs/task_a_trainval_test/predictions.csv")
pred_path.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"id": test["id"], "label": np.where(p > THRESH, "Hate", "Non-Hate")}).to_csv(pred_path, index=False)
# Restore the repository's original CodaBench input paths.
shutil.move(str(CANON_TRAIN.with_suffix(".csv.before_final")), str(CANON_TRAIN))
shutil.move(str(CANON_TEST.with_suffix(".csv.before_final")), str(CANON_TEST))
print("Task A test predictions:", len(pd.read_csv(pred_path)))


## 4. Package predictions for the released Task A test data

In [ ]:
ZIP = pathlib.Path("/kaggle/working/task_a_trainval_test.zip")
run([sys.executable, "-m", "hastika.common.submission", "--task", "a", "--pred", str(pred_path), "--inputs", str(test_path), "--out", str(ZIP)])
with zipfile.ZipFile(ZIP) as z: assert z.namelist() == ["predictions.csv"]
OUT = pathlib.Path("/kaggle/working/task_a_trainval_test_outputs")
OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(ZIP, OUT / ZIP.name); shutil.copy2(pred_path, OUT / pred_path.name)
json.dump({"train_rows_raw": len(combined), "train_rows_after_dedupe": len(fit_df), "w_svm": W_SVM, "threshold": THRESH}, open(OUT / "config.json", "w"), indent=2)
print("READY TO DOWNLOAD:", ZIP)
